# The Ultimate Guide to PyTrends: the Google Trends API with Python (2026 refresh)

Companion notebook to the MLforSEO guide by **Lazarina Stoy**.

This notebook shows you how to pull Google Trends data with Python: keyword suggestions, interest over time, interest by region, hourly interest, related queries and topics, and what is trending right now. It is written to run top to bottom in **Google Colab** (or any Jupyter environment).

**Read this first (October 2026 status):**

* **PyTrends** (`pip install pytrends`, v4.9.2) is *unofficial* and its GitHub repo was **archived in April 2025**. It still works for the core methods (`suggestions`, `interest_over_time`, `interest_by_region`, `related_queries`, `categories`), but it gets **rate-limited (HTTP 429)** quickly, and `related_topics` often crashes with an `IndexError` (we fall back to TrendSpy).
* `trending_searches`, `today_searches`, `realtime_trending_searches` and `top_charts` now return **404**: Google retired those endpoints when it launched the "Trending now" page. We use **TrendSpy** (`pip install trendspy`) or the official Trending now **RSS feed** for these instead.
* `get_historical_interest` was **removed** from PyTrends (it raises `NotImplementedError`). We use `interest_over_time` with an explicit hourly timeframe instead.
* PyTrends' built-in `retries=` / `backoff_factor=` options **crash** on current `urllib3` (2.x). Don't use them; use the `polite()` helper below.
* Google launched an **official Google Trends API (alpha)** in July 2025. Access is application-only. If you get in, prefer it over everything in this notebook: https://developers.google.com/search/apis/trends

Every request below goes to Google's undocumented web endpoints, so be gentle: small batches, pauses between calls, and cache what you pull.

## 1. Install and import the libraries

In [ ]:
# Colab already has pandas and matplotlib; this adds the two Trends libraries.
!pip install -q pytrends trendspy

In [ ]:
import random
import time
import warnings

import pandas as pd
import matplotlib.pyplot as plt

from pytrends.request import TrendReq
from pytrends.exceptions import ResponseError, TooManyRequestsError
from trendspy import Trends
from trendspy.client import TrendsQuotaExceededError

# pandas 2.x prints a FutureWarning from inside pytrends (fillna downcasting). It's harmless.
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 20)

## 2. Connect to Google Trends

`TrendReq` is the PyTrends connection object.

* `hl` is the interface language (`en-US`, `de`, `bg`...).
* `tz` is the timezone offset **in minutes behind UTC** (e.g. `0` for UTC, `360` for US Central, `-60` for Central European Time). It only changes how timestamps are labelled.
* `timeout=(connect, read)` in seconds. The default `(2, 5)` is too short and causes random `ReadTimeout`s, so raise it.
* **Do not** pass `retries=` or `backoff_factor=`: on urllib3 2.x PyTrends crashes with `TypeError: ... 'method_whitelist'`.

In [ ]:
pytrend = TrendReq(hl="en-US", tz=0, timeout=(10, 25))

# TrendSpy client: used for "Trending now" and as a fallback for related queries.
# request_delay adds a pause between its own requests.
tr = Trends(language="en", request_delay=4.0)

### The `polite()` helper: retries with exponential backoff

Google answers too many requests from one IP with **HTTP 429 (Too Many Requests)**. There is no published quota; in practice a handful of requests in quick succession can trigger it, and `related_queries` is the most sensitive endpoint.

`polite()` wraps any call: if Google says 429, it waits (30s, 60s, 120s... plus a little random jitter) and tries again. If it still fails after `retries` attempts it returns `None` rather than crashing the notebook, so you can carry on with the cells that did work.

If you keep hitting 429s: wait 15 to 60 minutes, restart the runtime (Colab gives you a new IP more often than not), or move to the official API / a paid provider.

In [ ]:
RATE_LIMIT_ERRORS = (TooManyRequestsError, TrendsQuotaExceededError)


def polite(fn, *args, retries=4, base_wait=30, pause=5, **kwargs):
    # Call fn(*args, **kwargs); back off and retry on HTTP 429.
    for attempt in range(retries):
        try:
            result = fn(*args, **kwargs)
            time.sleep(pause)  # breathing room before the next request
            return result
        except RATE_LIMIT_ERRORS as e:
            wait = base_wait * (2 ** attempt) + random.uniform(0, 5)
            print(f"429 / quota hit ({type(e).__name__}). Waiting {wait:.0f}s, attempt {attempt + 1}/{retries}")
            time.sleep(wait)
        except ResponseError as e:
            # 400 = bad request (e.g. keyword > 100 chars), 404 = retired endpoint. Retrying won't help.
            print(f"Google returned an error that retrying won't fix: {e}")
            return None
    print("Still rate-limited. Try again later, or restart the runtime for a fresh IP.")
    return None

## 3. Keyword suggestions (the "Add a search term" box)

`suggestions()` returns what the Google Trends search box suggests. Each result has a `mid` (a Knowledge Graph entity ID), a `title` and a `type`. The `mid` matters: you can pass it to `build_payload` instead of a plain search term to analyse a **topic** rather than an exact **search term** (see section 9).

Fun fact: in October 2021 the entity `/m/0hmyfsv` came back as "Facebook, Social networking service company". Run it today and the same entity comes back as **Meta**. Entities survive rebrands; keyword strings don't.

In [ ]:
keywords = polite(pytrend.suggestions, keyword="Facebook")
suggestions_df = pd.DataFrame(keywords)
suggestions_df.head(5)

## 4. Interest over time

Numbers represent search interest **relative to the highest point on the chart** for the given region and time (100 = the peak, 50 = half as popular, 0 = not enough data).

`build_payload()` sets up the request. Its parameters:

| Parameter | What it does | Example |
|---|---|---|
| `kw_list` | Up to **5** search terms or topic `mid`s | `["Netflix", "Disney+"]` |
| `timeframe` | `today 5-y` (default), `today 12-m`, `today 3-m`, `today 1-m`, `now 7-d`, `now 1-d`, `now 4-H`, `now 1-H`, `all` (since 2004), or explicit dates `"2025-01-01 2025-12-31"` | `"today 12-m"` |
| `geo` | Country / region code, empty = worldwide | `"US"`, `"GB"`, `"US-CA"` |
| `cat` | Category ID (0 = all). See `pytrend.categories()` | `5` (Computers & Electronics) |
| `gprop` | Google property: `""` (web search), `"images"`, `"news"`, `"youtube"`, `"froogle"` (Google Shopping) | `"youtube"` |

The granularity depends on the timeframe: about 1 to 7 days gives hourly data, up to ~9 months gives daily data, up to 5 years gives weekly data, longer gives monthly.

In [ ]:
kw_list = ["Facebook", "Apple", "Amazon", "Netflix", "Google"]

pytrend.build_payload(kw_list, timeframe="today 12-m", geo="")
iot = polite(pytrend.interest_over_time)

if iot is not None:
    # isPartial flags the latest, still-incomplete period; drop it before analysing
    iot = iot.drop(columns="isPartial", errors="ignore")
    display(iot.tail())
    iot.plot(figsize=(14, 6), title="Interest over time, past 12 months (worldwide)")
    plt.ylabel("Relative search interest (0-100)")
    plt.show()

### Hourly interest (the replacement for `get_historical_interest`)

The old `get_historical_interest()` method used to stitch together hourly data week by week. It was removed from PyTrends because the stitched values were not comparable across weeks (each chunk is scaled to its own 0-100). Today the honest way to get hourly data is to ask for a window of **7 days or less**, either with `now 7-d` or with explicit hours in the format `YYYY-MM-DDTHH YYYY-MM-DDTHH`.

In [ ]:
pytrend.build_payload(["Netflix", "Disney+"], timeframe="now 7-d")
hourly = polite(pytrend.interest_over_time)

if hourly is not None:
    hourly = hourly.drop(columns="isPartial", errors="ignore")
    hourly.plot(subplots=True, figsize=(14, 6), title="Hourly interest, last 7 days")
    plt.show()

# Explicit hourly window (max 7 days), e.g.:
# pytrend.build_payload(["Netflix"], timeframe="2026-10-01T00 2026-10-07T23")

### Long daily series (stitched safely)

Need *daily* data for more than ~9 months, e.g. as input for forecasting? PyTrends ships `dailydata.get_daily_data()`, which pulls month by month and rescales each month against a monthly series so the values are comparable. It makes one request per month, so expect 429s on long ranges: keep `wait_time` generous.

In [ ]:
from pytrends import dailydata

# Uncomment to run: 6 months = ~7 requests. Columns: <kw>_unscaled, <kw>_monthly, isPartial, scale, <kw>
# daily = dailydata.get_daily_data("netflix", 2026, 3, 2026, 8, geo="US", wait_time=10)
# daily[["netflix"]].plot(figsize=(14, 5))

### A cool mini-project: share of search

If your keywords are competitors, interest over time lets you compute **share of search**: each brand's interest as a percentage of the total for that period. It is a decent proxy for share of voice/market, and a great input for forecasting.

In [ ]:
if iot is not None:
    share_of_search = iot[kw_list].div(iot[kw_list].sum(axis=1), axis=0) * 100
    share_of_search.plot.area(figsize=(14, 6), title="Share of search (%)")
    plt.ylabel("% of combined interest")
    plt.show()
    display(share_of_search.mean().round(1).sort_values(ascending=False).rename("avg share %"))

## 5. Interest by region

Examines search term popularity by location during the timeframe set in `build_payload`. Values are calculated on a scale from 0 to 100, where **100 is the location with the most popularity as a fraction of total searches in that location**. A higher value means a higher *proportion* of all queries, not a higher absolute count.

* `resolution`: `"COUNTRY"`, `"REGION"` (states/provinces), `"CITY"`, or `"DMA"` (US metro areas, only with `geo="US"`).
* `inc_low_vol=True` includes low-search-volume locations (more rows, more zeros).
* `inc_geo_code=True` adds the ISO code column, handy for maps.

In [ ]:
pytrend.build_payload(kw_list, timeframe="today 1-m")
regiondf = polite(pytrend.interest_by_region, resolution="COUNTRY", inc_low_vol=False, inc_geo_code=True)

if regiondf is not None:
    # keep only the countries where every keyword has a non-zero score
    regiondf = regiondf[(regiondf[kw_list] != 0).all(axis=1)]
    regiondf = regiondf.dropna(how="all", axis=0)
    display(regiondf.head(10))

    regiondf.plot(figsize=(20, 10), y=kw_list, kind="bar", title="Interest by country, past month")
    plt.ylabel("Relative interest (0-100)")
    plt.show()

## 6. Related queries

"Users searching for your term also searched for these queries." For each keyword you get two tables:

* **top**: the most popular related queries (100 = the most common one).
* **rising**: queries with the biggest increase since the previous period. `Breakout` means growth of more than 5,000% (usually brand-new queries).

This is the endpoint Google rate-limits hardest. We try PyTrends first and fall back to TrendSpy. Unlike the 2021 version of this code (which only kept the first keyword), this loops over **every** keyword and stacks the results into one tidy table.

In [ ]:
def related_queries_table(keywords, timeframe="today 3-m", geo=""):
    # One tidy DataFrame: keyword | type (top/rising) | query | value
    rows = []
    pytrend.build_payload(keywords, timeframe=timeframe, geo=geo)
    result = polite(pytrend.related_queries)

    if result is None:  # fallback: TrendSpy, one keyword at a time
        print("PyTrends was rate-limited, trying TrendSpy...")
        result = {}
        for kw in keywords:
            r = polite(tr.related_queries, kw, timeframe=timeframe, geo=geo,
                       headers={"referer": "https://www.google.com/"})
            if r is not None:
                result[kw] = r

    for kw, tables in (result or {}).items():
        for kind in ("top", "rising"):
            df = tables.get(kind)
            if df is not None and not df.empty:
                df = df.copy()
                df.insert(0, "type", kind)
                df.insert(0, "keyword", kw)
                rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


allqueries = related_queries_table(["Netflix", "Disney+"])
allqueries.head(30)

Prefer the side-by-side layout from the original guide (top queries next to rising queries)? Pivot it:

In [ ]:
if not allqueries.empty:
    side_by_side = pd.concat(
        [
            allqueries[allqueries.type == "top"][["keyword", "query", "value"]]
            .rename(columns={"query": "top query", "value": "top query value"})
            .reset_index(drop=True),
            allqueries[allqueries.type == "rising"][["query", "value"]]
            .rename(columns={"query": "rising query", "value": "rising query value"})
            .reset_index(drop=True),
        ],
        axis=1,
    )
    display(side_by_side.head(25))

## 7. Related topics

Same idea, but returns **topics** (entities) rather than query strings: a title, a topic type and a `mid` you can feed straight back into `build_payload`.

Heads-up: in October 2026, PyTrends' `related_topics()` regularly crashes with `IndexError: list index out of range` (Google sometimes sends back an empty list, and PyTrends only expects a missing one). The function below catches that and falls back to TrendSpy.

In [ ]:
def related_topics_table(keyword, timeframe="today 3-m", geo=""):
    # Top + rising related topics for ONE keyword: keyword | type | title | topic type | mid | value
    tables = None
    try:
        pytrend.build_payload([keyword], timeframe=timeframe, geo=geo)
        result = polite(pytrend.related_topics)  # dict of DataFrames
        if result and result.get(keyword):
            t = result[keyword]
            rename = {"topic_title": "title", "topic_type": "topic_type", "topic_mid": "mid"}
            tables = {k: (v.rename(columns=rename) if v is not None else None) for k, v in t.items()}
    except IndexError:
        # pytrends 4.9.2 bug: Google sometimes returns an empty list, which pytrends doesn't handle
        print("PyTrends couldn't parse the response, trying TrendSpy...")

    if not tables:
        r = polite(tr.related_topics, keyword, timeframe=timeframe, geo=geo,
                   headers={"referer": "https://www.google.com/"})
        if r:
            tables = {k: v.rename(columns={"type": "topic_type"}) for k, v in r.items()}

    rows = []
    for kind in ("top", "rising"):
        df = (tables or {}).get(kind)
        if df is not None and not df.empty:
            df = df[[c for c in ["title", "topic_type", "mid", "value", "formattedValue"] if c in df.columns]].copy()
            df.insert(0, "type", kind)
            df.insert(0, "keyword", keyword)
            rows.append(df)
    if not rows:
        print("No related topics returned (often a sign of rate-limiting). Try again in a few minutes.")
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


related_topic = related_topics_table("Netflix")
related_topic.head(20)

## 8. What's trending right now (replaces `trending_searches`, `today_searches`, `realtime_trending_searches`)

In 2024 Google replaced "Daily search trends" and "Realtime search trends" with a single **Trending now** page. The old PyTrends methods that read those pages now return HTTP 404:

In [ ]:
# Kept for reference: these now fail with "Google returned a response with code 404"
for name, call in [("trending_searches", lambda: pytrend.trending_searches(pn="united_states")),
                   ("today_searches", lambda: pytrend.today_searches(pn="US")),
                   ("realtime_trending_searches", lambda: pytrend.realtime_trending_searches(pn="US")),
                   ("top_charts", lambda: pytrend.top_charts(2024, hl="en-US", tz=300, geo="GLOBAL"))]:
    try:
        call()
        print(f"{name}: works again!")
    except ResponseError as e:
        print(f"{name}: {e}")

### Option A: TrendSpy `trending_now()`

Returns the full Trending now list (hundreds of trends for the past 24 hours, or `hours=4`, `48`, `168`), each with an approximate search volume, growth, start time, related breakdown keywords and topics.

In [ ]:
trending = polite(tr.trending_now, geo="US", hours=24)

if trending:
    trending_df = pd.DataFrame(
        {
            "keyword": t.keyword,
            "search_volume": t.volume,
            "growth_pct": t.volume_growth_pct,
            "started_utc": pd.to_datetime(t.started_timestamp[0], unit="s", utc=True) if t.started_timestamp else None,
            "active": not t.is_trend_finished,
            "topics": ", ".join(t.topic_names),
            "breakdown": ", ".join((t.trend_keywords or [])[:5]),
        }
        for t in trending
    ).sort_values("search_volume", ascending=False)
    display(trending_df.head(20))

### Option B: the official Trending now RSS feed (no library needed)

`https://trends.google.com/trending/rss?geo=US` is a public feed with the top ~10 to 20 current trends, an approximate traffic band and linked news articles. It's the most stable option because it's a documented feed rather than an internal endpoint.

In [ ]:
import requests
import xml.etree.ElementTree as ET

def trending_rss(geo="US"):
    url = f"https://trends.google.com/trending/rss?geo={geo}"
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
    resp.raise_for_status()
    ns = {"ht": "https://trends.google.com/trending/rss"}
    root = ET.fromstring(resp.content)
    return pd.DataFrame(
        {
            "title": item.findtext("title"),
            "approx_traffic": item.findtext("ht:approx_traffic", namespaces=ns),
            "published": item.findtext("pubDate"),
            "top_news": item.findtext("ht:news_item/ht:news_item_title", namespaces=ns),
        }
        for item in root.iter("item")
    )

trending_rss("US").head(20)

### What about Top Charts / Year in Search?

`top_charts()` (yearly top searches) is gone too: the endpoint returns 404 and there is no working replacement in PyTrends or TrendSpy. The yearly lists now live on Google's **Year in Search** pages (https://trends.withgoogle.com/year-in-search/2025/), which you can browse or export manually.

## 9. Working around the limitations

### Search terms vs topics
`"apple"` the search term matches the literal string; the topic `Apple Inc.` (a `mid` from `suggestions()`) groups all searches about the company in every language and spelling. Don't mix the two in one comparison. Here's how to request a topic:

In [ ]:
apple_options = pd.DataFrame(polite(pytrend.suggestions, keyword="Apple"))
display(apple_options)

# pick the company entity, not the fruit
apple_inc_mid = apple_options.loc[apple_options["type"].str.contains("company", case=False), "mid"].iloc[0]
pytrend.build_payload([apple_inc_mid], timeframe="today 12-m")
apple_topic = polite(pytrend.interest_over_time)
if apple_topic is not None:
    apple_topic.drop(columns="isPartial", errors="ignore").plot(figsize=(14, 4), title=f"Topic: Apple Inc. ({apple_inc_mid})")
    plt.show()

### Comparing more than five keywords: the anchor-term method

Google Trends only compares five entries at a time, and every request is scaled to its own 0-100, so you can't just glue two requests together. The workaround: put the **same anchor keyword** in every batch (four new keywords + the anchor), then rescale each batch so the anchor matches its value in the first batch. Pick an anchor with steady, mid-sized interest; if it's tiny, rounding noise gets amplified.

In [ ]:
def compare_many(keywords, anchor, timeframe="today 12-m", geo=""):
    # Mean interest for any number of keywords, on one comparable scale (anchor = reference).
    others = [k for k in keywords if k != anchor]
    batches = [others[i:i + 4] for i in range(0, len(others), 4)]
    combined, reference = {}, None

    for batch in batches:
        pytrend.build_payload(batch + [anchor], timeframe=timeframe, geo=geo)
        df = polite(pytrend.interest_over_time)
        if df is None:
            print(f"Skipped batch {batch} (rate-limited)")
            continue
        means = df.drop(columns="isPartial", errors="ignore").mean()
        if reference is None:
            reference = means[anchor]
        scale = reference / means[anchor] if means[anchor] else float("nan")
        for kw in batch:
            combined[kw] = means[kw] * scale
    if reference is None:
        print("Every batch was rate-limited. Wait a while and try again.")
        return pd.Series(dtype=float)
    combined[anchor] = reference
    return pd.Series(combined).sort_values(ascending=False).rename("relative interest (anchor-scaled)")


streaming = ["Netflix", "Disney+", "Hulu", "Max", "Peacock", "Paramount+", "Apple TV", "Prime Video", "Crunchyroll"]
compare_many(streaming, anchor="Hulu", geo="US")

## 10. Save and download your data

In [ ]:
if not allqueries.empty:
    allqueries.to_csv("allqueries.csv", index=False)
if iot is not None:
    iot.to_csv("interest_over_time.csv")

# Download from Colab (skipped automatically outside Colab)
try:
    from google.colab import files
    for f in ["allqueries.csv", "interest_over_time.csv"]:
        try:
            files.download(f)
        except FileNotFoundError:
            pass
except ImportError:
    print("Not running in Colab: files saved to the current folder.")

## 11. The official Google Trends API (alpha)

In July 2025 Google announced an official Google Trends API, in alpha. What Google has said publicly:

* a rolling **five-year** window of data, up to about two days ago;
* **daily, weekly, monthly and yearly** aggregations;
* **country and sub-region** (ISO 3166-2) breakdowns;
* **consistently scaled** data, so results from different requests can be compared and joined (the thing PyTrends can't do).

Access is by application on a rolling basis, and there's no public self-serve key, price or quota table yet. Apply at https://developers.google.com/search/apis/trends. If you get access, use it for anything long-running or client-facing, and keep this notebook for quick exploration.

---
Made with care by [MLforSEO](https://www.mlforseo.com/). Learn more in the [MLforSEO Academy](https://academy.mlforseo.com/).